# 01 - 语雀 (Yuque) 内部 Web API 完整测试

> 本 Notebook 用于测试语雀内部 Web API 的全部 CRUD 功能。
> 使用 Cookie 认证，完全免费，无需超级会员。

---

## 📋 目录

1. [环境准备](#环境准备) — 加载 Cookie、构建请求头
2. [列出知识库](#1-列出知识库) — `GET /api/books`
3. [获取目录](#2-获取目录) — `GET /api/books/{id}/toc`
4. [读取文档](#3-读取文档) — `GET /api/docs/{slug}?book_id={id}`
5. [创建文档](#4-创建文档) — `POST /api/docs`
6. [更新文档](#5-更新文档) — `PUT /api/docs/{id}`
7. [删除文档](#6-删除文档) — `DELETE /api/docs/{id}`
8. [附录：端点总结](#附录端点总结)

---

## 🔐 认证方式说明

语雀有两套 API：

| API 体系 | 认证方式 | 需要会员 | 端点前缀 |
|----------|----------|----------|----------|
| **Open API v2** | `X-Auth-Token` | ✅ 超级会员 | `/api/v2/...` |
| **内部 Web API** | `Cookie` | ❌ **完全免费** | `/api/...` |

本 Notebook 使用**内部 Web API**，通过浏览器 Cookie 认证。

### 获取 Cookie 步骤

1. 登录语雀网页版：https://www.yuque.com
2. 按 **F12** 打开浏览器开发者工具
3. 切换到 **Application**（应用）标签
4. 左侧点击 **Cookies** → `https://www.yuque.com`
5. 复制以下两个值：
   - `_yuque_session` 的 **Value**
   - `_ctoken` 的 **Value**
6. 粘贴到 `.env` 文件或下方代码中

> ⚠️ **安全提醒**：这两个值是你的登录凭证，等同于账号密码。**不要泄露给他人，不要提交到 Git！**

---

## 环境准备

加载依赖、读取 Cookie、定义通用请求函数。

In [ ]:
# =============================================================================
# 环境准备：加载 Cookie 和依赖
# =============================================================================

import os
import json
import requests
from pathlib import Path
from dotenv import load_dotenv

# 尝试从 .env 加载（如果文件编码有问题则手动填写）
try:
    load_dotenv(Path('../../.env'))
except Exception as e:
    print(f'加载 .env 失败（可忽略）: {e}')

# =============================================================================
# 【配置区】请在这里填写你的 Cookie，或确保 .env 中有 YUQUE_SESSION 和 YUQUE_CTOKEN
# =============================================================================
SESSION = os.getenv('YUQUE_SESSION') or '你的_yuque_session'
CTOKEN  = os.getenv('YUQUE_CTOKEN')  or '你的_ctoken'

BASE = 'https://www.yuque.com'
COOKIE = f'_yuque_session={SESSION}; _ctoken={CTOKEN}'

print(f'✅ 配置完成: SESSION长度={len(SESSION)}, CTOKEN长度={len(CTOKEN)}')


def build_headers(referer=None):
    """
    构建 HTTP 请求头。
    
    【注意】写操作（POST/PUT/DELETE）必须提供 referer 参数，
    否则语雀会返回 403 "missing csrf referer or origin" 错误。
    """
    h = {
        'Cookie': COOKIE,
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'Accept': 'application/json, text/plain, */*',
        'X-CSRF-Token': CTOKEN,          # CSRF 防护令牌（_ctoken 的值）
        'X-Requested-With': 'XMLHttpRequest',  # 标记为 Ajax 请求
    }
    if referer:
        h['Referer'] = referer           # 写操作必需！
    return h


def api(method, path, data=None, query=None, referer=None):
    """
    调用语雀内部 Web API 的通用函数。
    
    参数:
        method:  'GET' | 'POST' | 'PUT' | 'DELETE'
        path:    API 路径（不含域名前缀，如 '/api/books'）
        data:    请求体（POST/PUT 时使用）
        query:   URL 查询参数（字典）
        referer: Referer 头（写操作必需）
    """
    url = f'{BASE}{path}'
    if query:
        url += '?' + '&'.join(f'{k}={v}' for k, v in query.items())
    
    h = build_headers(referer)
    
    # 写操作需要 Content-Type
    if data and method not in ('GET', 'DELETE'):
        h['Content-Type'] = 'application/json'
    
    if method == 'GET':
        r = requests.get(url, headers=h)
    elif method == 'POST':
        r = requests.post(url, headers=h, json=data)
    elif method == 'PUT':
        r = requests.put(url, headers=h, json=data)
    elif method == 'DELETE':
        r = requests.delete(url, headers=h)
    
    return r.json()

print('✅ 通用函数定义完成！')

---

## 1. 列出知识库

**端点**：`GET /api/books`

**说明**：返回当前登录用户的所有知识库列表。

**响应示例**：
```json
{
  "data": [
    {"id": 68025057, "slug": "qah8x7", "name": "LLM知识库", ...},
    {"id": 68016047, "slug": "zf1hbk", "name": "Awesome-CS336", ...}
  ]
}
```

> 💡 **提示**：记下第一个知识库的 ID，后续测试会用到。

In [ ]:
# =============================================================================
# 1. 列出知识库
# =============================================================================

r = api('GET', '/api/books')
books = r.get('data', [])

print(f'📚 知识库数量: {len(books)}')
print()
for i, b in enumerate(books[:5]):
    print(f'  [{i+1}] ID:{b["id"]:<10} Slug:{b["slug"]:<15} Name:{b["name"]}')

# 保存第一个知识库 ID 用于后续测试
if books:
    BOOK_ID = books[0]['id']
    BOOK_SLUG = books[0]['slug']
    print()
    print(f'👉 后续测试使用知识库: ID={BOOK_ID}, Slug={BOOK_SLUG}')
else:
    print('⚠️ 未找到知识库，请检查 Cookie 是否正确')

---

## 2. 获取目录

**端点**：`GET /api/books/{id}/toc`

**说明**：返回知识库的目录结构（TOC），包含两类条目：
- `TITLE`：目录/分组项（类似于文件夹）
- `DOC`：实际文档项

**响应示例**：
```json
{
  "data": {
    "toc": [
      {"type": "TITLE", "title": "第一章", "depth": 0, ...},
      {"type": "DOC", "title": "概述", "url": "abc123", "depth": 1, ...}
    ]
  }
}
```

> 💡 **提示**：DOC 类型的 `url` 字段就是文档的 **slug**，读取文档时需要用到。

In [ ]:
# =============================================================================
# 2. 获取目录
# =============================================================================

r = api('GET', f'/api/books/{BOOK_ID}/toc')
toc = r.get('data', {}).get('toc', [])

print(f'📑 目录项数量: {len(toc)}')
print()
for item in toc[:15]:
    indent = '  ' * (item.get('depth', 0))
    icon = '📄' if item['type'] == 'DOC' else '📁'
    slug_info = f" (slug: {item['url']})" if item.get('url') else ''
    print(f'{indent}{icon} {item["title"]}{slug_info}')

# 找一个 DOC 类型的条目用于读取测试
doc_items = [i for i in toc if i['type'] == 'DOC']
if doc_items:
    DOC_SLUG = doc_items[0]['url']
    print()
    print(f'👉 使用文档 Slug: {DOC_SLUG}')
else:
    print('⚠️ 未找到文档类型条目')

---

## 3. 读取文档

**端点**：`GET /api/docs/{slug}?book_id={id}`

**说明**：读取文档的完整详情。

**响应中的关键字段**：
| 字段 | 说明 |
|------|------|
| `id` | 文档数字 ID（更新/删除必需）|
| `title` | 文档标题 |
| `slug` | 文档 URL 标识 |
| `content` | 文档内容（**Lake HTML 格式**）|
| `format` | 格式（通常为 `lake`）|

> ⚠️ **注意**：内容字段是 `content`，不是 `body` 或 `body_asl`。
> Lake 是语雀自研的富文本格式，基于 HTML，但包含自定义标签。

In [ ]:
# =============================================================================
# 3. 读取文档
# =============================================================================

r = api('GET', f'/api/docs/{DOC_SLUG}', query={'book_id': str(BOOK_ID)})
doc = r.get('data', {})

print(f'📄 文档详情:')
print(f'   标题: {doc.get("title")}')
print(f'   ID: {doc.get("id")}')
print(f'   Slug: {doc.get("slug")}')
print(f'   格式: {doc.get("format")}')
print(f'   字数: {doc.get("word_count")}')
print(f'   更新时间: {doc.get("updated_at")}')

content = doc.get('content', '')
print(f'   内容长度: {len(content)} 字符')
print()
print('--- 内容预览（前 800 字符）---')
print(content[:800])
print('...')

---

## 4. 创建文档

**端点**：`POST /api/docs`

**请求体**：
```json
{
  "book_id": 68025057,           // 知识库 ID（数字）
  "title": "文档标题",            // 文档标题
  "body": "<!doctype lake>...",  // 内容（Lake HTML 格式）
  "format": "lake",              // 格式：lake（固定）
  "public": 0                    // 可见性：0=私密, 1=公开, 2=空间公开
}
```

**⚠️ 重要提醒**：
- 写操作必须有 `Referer` 头，否则返回 **403**
- `body` 必须以 `<!doctype lake>` 开头
- 创建后文档**不会自动出现在目录中**，需手动在语雀网页版调整

**响应示例**：
```json
{
  "data": {
    "id": 266476793,              // 文档数字 ID
    "slug": "nxh6ktx04drq0uft",   // 文档 slug
    "title": "文档标题"
  }
}
```

In [ ]:
# =============================================================================
# 4. 创建文档
# =============================================================================

from datetime import datetime

# 构建请求体
create_payload = {
    'book_id': BOOK_ID,
    'title': f'API测试文档-{datetime.now().strftime("%H%M%S")}',
    # body 必须是 Lake 格式，以 <!doctype lake> 开头
    'body': '<!doctype lake><h1>测试标题</h1><p>这是一段测试内容。</p><ul><li>列表项1</li><li>列表项2</li></ul>',
    'format': 'lake',
    'public': 0,  # 0=私密
}

print('📤 发送创建请求...')
r = api('POST', '/api/docs', data=create_payload, referer=f'{BASE}/{BOOK_ID}')

if 'data' in r:
    created = r['data']
    TEST_DOC_ID = created['id']
    TEST_DOC_SLUG = created['slug']
    print(f'✅ 创建成功!')
    print(f'   ID: {TEST_DOC_ID}')
    print(f'   Slug: {TEST_DOC_SLUG}')
    print(f'   标题: {created["title"]}')
    print(f'   URL: https://www.yuque.com/{BOOK_SLUG}/{TEST_DOC_SLUG}')
else:
    print(f'❌ 创建失败:')
    print(json.dumps(r, ensure_ascii=False, indent=2))

---

## 5. 更新文档

**端点**：`PUT /api/docs/{id}`

**请求体**：
```json
{
  "title": "新标题",               // 新标题（可选）
  "body": "<!doctype lake>...",  // 新内容（可选）
  "format": "lake"                // 格式（固定 lake）
}
```

**⚠️ 重要提醒**：
- URL 中的 `{id}` 是**数字 ID**（如 `266476793`），不是 slug！
- 必须先调用读取接口获取 `doc_id`
- 写操作必须有 `Referer` 头

In [ ]:
# =============================================================================
# 5. 更新文档
# =============================================================================

# 构建请求体
update_payload = {
    'title': 'API测试文档 [已更新]',
    'body': '<!doctype lake><h1>更新后的标题</h1><p>这是更新后的内容。</p><p><strong>加粗文字</strong></p>',
    'format': 'lake',
}

print('📤 发送更新请求...')
print(f'   文档 ID: {TEST_DOC_ID}')
r = api('PUT', f'/api/docs/{TEST_DOC_ID}', data=update_payload, referer=f'{BASE}/{BOOK_ID}')

if 'data' in r:
    updated = r['data']
    print(f'✅ 更新成功!')
    print(f'   标题: {updated["title"]}')
    print(f'   更新时间: {updated.get("updated_at", "N/A")}')
else:
    print(f'❌ 更新失败:')
    print(json.dumps(r, ensure_ascii=False, indent=2))

---

## 6. 删除文档

**端点**：`DELETE /api/docs/{id}?book_id={book_id}`

**⚠️ 警告**：
- 删除操作**不可逆**！
- URL 中的 `{id}` 是**数字 ID**，不是 slug！
- 写操作必须有 `Referer` 头

> 💡 **测试建议**：本单元格默认不执行删除，如需测试请取消注释最后一行。

In [ ]:
# =============================================================================
# 6. 删除文档
# =============================================================================

print('⚠️  准备删除文档...')
print(f'   文档 ID: {TEST_DOC_ID}')
print(f'   文档 Slug: {TEST_DOC_SLUG}')
print()

# 【安全锁】默认注释掉删除操作，防止误删
# 如需测试删除，请取消下面两行的注释

# r = api('DELETE', f'/api/docs/{TEST_DOC_ID}', query={'book_id': str(BOOK_ID)}, referer=f'{BASE}/{BOOK_ID}')
# print('✅ 删除成功!' if 'data' in r else f'❌ 删除失败: {r}')

print('🔒 删除操作已跳过（代码被注释）。如需测试，请取消注释上述两行。')

---

## 附录：端点总结

### 读取操作（无需 CSRF）

| 方法 | 端点 | 说明 |
|------|------|------|
| GET | `/api/books` | 列出知识库 |
| GET | `/api/books/{id}/toc` | 获取目录 |
| GET | `/api/docs/{slug}?book_id={id}` | 读取文档 |

### 写操作（需要 `X-CSRF-Token` + `Referer`）

| 方法 | 端点 | Body | 说明 |
|------|------|------|------|
| POST | `/api/docs` | `{book_id, title, body, format, public}` | 创建文档 |
| PUT | `/api/docs/{id}` | `{title, body, format}` | 更新文档（id 是数字 ID）|
| DELETE | `/api/docs/{id}?book_id={id}` | - | 删除文档（id 是数字 ID）|

### 请求头模板

```python
headers = {
    'Cookie': '_yuque_session=xxx; _ctoken=xxx',
    'User-Agent': 'Mozilla/5.0 ...',
    'Accept': 'application/json, text/plain, */*',
    'X-CSRF-Token': 'xxx',              # _ctoken 的值
    'X-Requested-With': 'XMLHttpRequest',
    'Referer': 'https://www.yuque.com/{book_id}',  # 写操作必需！
}
```

### 常见错误码

| 状态码 | 错误信息 | 原因 |
|--------|----------|------|
| 401 | Unauthorized | Cookie 过期或无效 |
| 403 | missing csrf referer or origin | 缺少 Referer 头 |
| 404 | Not Found | 文档/知识库不存在 |
| 422 | id invalid | PUT 时缺少 id 参数 |

---

> 📌 **本文档位置**：`docs-internal/yuque-api-lab/01_yuque_webapi.ipynb`
> 📌 **后端路由**：`server/routes/yuque.ts`
> 📌 **前端工具**：`src/theme/tools/yuque/`
> 📌 **Agent Skill**：`.skills/yuque-assistant/SKILL.md`